In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

PROJECT_PATH = "/content/drive/MyDrive/AI_Resume_Analyzer_and_Job_Recommendation_System"

os.chdir(PROJECT_PATH)

print(os.getcwd())

/content/drive/.shortcut-targets-by-id/16N9xqXDRfgDbBpP92jzc4yE1m_O2Nwfi/AI_Resume_Analyzer_and_Job_Recommendation_System


In [3]:
!pip install -q transformers datasets accelerate evaluate torch scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [4]:
from src.config import (
    RESUME_MODEL_VERSION,
    TRAIN_CSV,
    TEST_CSV,
    MODEL_DIR
)

print("=" * 50)
print("Resume Model Version :", RESUME_MODEL_VERSION)
print("Train CSV            :", TRAIN_CSV)
print("Test CSV             :", TEST_CSV)
print("Model Directory      :", MODEL_DIR)
print("=" * 50)

Resume Model Version : resume_v2
Train CSV            : /content/drive/.shortcut-targets-by-id/16N9xqXDRfgDbBpP92jzc4yE1m_O2Nwfi/AI_Resume_Analyzer_and_Job_Recommendation_System/data/processed/train_resume_2.csv
Test CSV             : /content/drive/.shortcut-targets-by-id/16N9xqXDRfgDbBpP92jzc4yE1m_O2Nwfi/AI_Resume_Analyzer_and_Job_Recommendation_System/data/processed/test_resume_2.csv
Model Directory      : /content/drive/.shortcut-targets-by-id/16N9xqXDRfgDbBpP92jzc4yE1m_O2Nwfi/AI_Resume_Analyzer_and_Job_Recommendation_System/saved_models/resume_v2


In [5]:
import os
from src.config import MODEL_DIR

os.makedirs(
    str(MODEL_DIR / "roberta"),
    exist_ok=True
)

print("Directory created successfully.")

Directory created successfully.


In [6]:
import pandas as pd
from src.config import TRAIN_CSV, TEST_CSV

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train Shape:", train_df.shape)
print("Test Shape :", test_df.shape)

train_df.head()

Train Shape: (8000, 2)
Test Shape : (2000, 2)


,text,label
0,Education: Web Design Certification Experience...,8
1,Education: Bachelor's in Marketing Experience:...,25
2,Education: MBA Experience: 10 years Skills: Te...,25
3,Education: Master's in Healthcare Administrati...,17
4,Education: Bachelor's in Computer Science Expe...,9


In [7]:
!python -m src.models.roberta.train

Loading processed datasets...
Train Shape: (8000, 2)
Test Shape : (2000, 2)
Loading tokenizer...
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 110kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 11.3MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 3.66MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 44.2MB/s]
Loading RoBERTa model...
config.json: 100% 481/481 [00:00<00:00, 2.43MB/s]
model.safetensors: 100% 499M/499M [00:02<00:00, 177MB/s]
Loading weights: 100% 197/197 [00:00<00:00, 5443.09it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      |

In [8]:
from src.config import MODEL_DIR

print("Saved files:")

!ls "{MODEL_DIR}/roberta"

Saved files:
checkpoint-1000  checkpoint-2500  model.safetensors	 training_args.bin
checkpoint-1500  checkpoint-500   tokenizer_config.json
checkpoint-2000  config.json	  tokenizer.json


In [9]:
!python -m src.models.roberta.evaluate

Loading weights: 100% 201/201 [00:00<00:00, 2658.38it/s]
100% 250/250 [00:05<00:00, 47.84it/s]
ROBERTA MODEL EVALUATION
Test Samples     : 2000
Number of Classes: 42

===== ROBERTA RESULTS =====
Accuracy : 0.9880
Precision: 0.9886
Recall   : 0.9880
F1 Score : 0.9880

Classification Report
              precision    recall  f1-score   support

           0     0.8966    1.0000    0.9455        26
           1     1.0000    1.0000    1.0000        23
           2     1.0000    1.0000    1.0000        29
           3     1.0000    0.9000    0.9474        20
           4     0.9714    1.0000    0.9855        34
           5     1.0000    0.9474    0.9730        19
           6     1.0000    0.9375    0.9677        32
           7     0.9630    0.9630    0.9630        27
           8     0.9710    1.0000    0.9853        67
           9     0.9580    1.0000    0.9785       114
          10     1.0000    1.0000    1.0000        37
          11     1.0000    1.0000    1.0000        18
       

In [10]:
from src.models.roberta.predict import ResumePredictor

predictor = ResumePredictor()

resume = """
Experienced Machine Learning Engineer with Python,
TensorFlow, SQL, Deep Learning,
Computer Vision and NLP.
"""

prediction = predictor.predict(resume)

print(prediction)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'category': 'Technology', 'confidence': 0.9993}


In [11]:
import os
from src.config import MODEL_DIR

model_path = MODEL_DIR / "roberta"

size = 0

for path, dirs, files in os.walk(model_path):
    for f in files:
        fp = os.path.join(path, f)
        size += os.path.getsize(fp)

print(f"Model Size: {size/(1024*1024):.2f} MB")

Model Size: 7614.24 MB


In [12]:
import pandas as pd
from src.config import TRAIN_CSV, TEST_CSV

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

# Normalize text
train_text = (
    train_df["text"]
    .astype(str)
    .str.strip()
    .str.lower()
)

test_text = (
    test_df["text"]
    .astype(str)
    .str.strip()
    .str.lower()
)

duplicates = set(train_text).intersection(set(test_text))

print("=" * 50)
print("Train Samples :", len(train_df))
print("Test Samples  :", len(test_df))
print("Duplicate Resumes :", len(duplicates))
print("=" * 50)

Train Samples : 8000
Test Samples  : 2000
Duplicate Resumes : 0


In [13]:
import pandas as pd
from src.config import TRAIN_CSV

train_df = pd.read_csv(TRAIN_CSV)

print("Number of Classes:", train_df["label"].nunique())

print("\nSamples per Class:")
print(train_df["label"].value_counts().sort_index())

Number of Classes: 42

Samples per Class:
label
0      106
1       93
2      118
3       81
4      136
5       78
6      127
7      107
8      267
9      454
10     146
11      73
12     348
13     155
14      47
15     202
16      72
17     390
18     147
19      86
20     210
21      78
22      54
23     129
24     196
25     370
26     102
27      78
28      79
29     213
30     129
31      69
32     180
33      94
34     112
35     223
36      60
37     110
38    2009
39     122
40      98
41      52
Name: count, dtype: int64
